In [16]:
# Importing Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
from sklearn.metrics import mean_squared_error, r2_score
%run functions.ipynb

In [17]:
# Check active directory
os.getcwd()

'C:\\Users\\Lenovo\\OneDrive - Ashesi University\\Desktop\\Code_Folder\\Solar_BESS_Optimal_Sizing'

In [18]:
# Loading the weather data and solar PV production data as a panda datafrme

weather_data = pd.read_excel('Complete_data_01.xlsx')

In [22]:
# Loading an example of a typical load demand data as obtained from the solar PV monitoring system

load_demand = pd.read_excel('example_load_profile_data.xlsx')['Hourly Power Demand kw']

In [21]:
# weather data

weather_data.head(5)

,Datetime,GHI ALL SKY,CLEAR SKY,DNI,DHI,Albedo,T2M WET,WS10M,Internal_Production
0,2023-11-01 00:00:00,0.0,0.0,0.0,0.0,-999.0,24.52,1.85,0.0
1,2023-11-01 01:00:00,0.0,0.0,0.0,0.0,-999.0,24.37,1.80,0.0
2,2023-11-01 02:00:00,0.0,0.0,0.0,0.0,-999.0,24.22,1.77,0.0
3,2023-11-01 03:00:00,0.0,0.0,0.0,0.0,-999.0,24.07,1.77,0.0
4,2023-11-01 04:00:00,0.0,0.0,0.0,0.0,-999.0,23.93,1.87,0.0


In [23]:
# Example load demand data

load_demand

0       101.73125
1        96.50625
2        92.01875
3       137.42500
4       122.75000
          ...    
3679    158.27500
3680    212.35625
3681    230.57500
3682    396.28750
3683    426.90625
Name: Hourly Power Demand kw, Length: 3684, dtype: float64

In [6]:
# Extracting parameters from the data into arrays

n_day = np.array(weather_data['Datetime'].dt.dayofyear)           # Numbers of days array

day_hours = np.array(weather_data['Datetime'].dt.hour)            # Hours in a day (1 - 24) array

ghi = np.array(weather_data['GHI ALL SKY'])                       # Global horizontal irradiance array

temp = np.array(weather_data['T2M WET'])                          # Ambient temperature array

internal_prod_ = np.array(weather_data['Internal_Production'])    # solar pv hourly power production array

In [ ]:
# Solar PV orientation and quantity of the engineering building roof at Ashesi Campus

Engineering_Building_panels_azimuth = np.array([253.57, 75.22, 294.17, 111.85, 343, 159.21])
Panels_per_azimuth = np.array([25, 32, 60, 81, 12, 28])

In [7]:
# Input parameters for the single solar pv diode model

Np = 4                     # number of parallel solar pv strings per inverter                   
Ns = 60                    # number of solar cells in the existing solar pv panel on the campus
N_panels_in_string = 20    # number of solar pv panels in a string
Rs = 1.211e-01             # model series resistance
Rsh = 0.111e+01            # model parallel resistance
Io = 9.99e-13              # model reverse saturation current
n = 5.5                    # model ideality factor
q = 1.6021e-19             # electron charge
K = 1.3806e-23             # boltzmann constant

pv_powerModel_inputs = [n_day, day_hours, values, 10, Engineering_Building_panels_azimuth, Panels_per_azimuth, ghi, temp, 0.2]
pv_ivModel_inputs = [4, 60, 20, Rs, Rsh, Io, n, q, K, temp]

In [8]:
# Input parameters for probabilistic grid outage model simulation

N_monte_carlo_trials = 100      # number of monte_carlo trials
transformer_power_kVA = 200     # assumed transformer rating (Capacity)
power_factor = 0.8              # assumed transformer power factor
avg_eff = 0.8                   # assumed transformer efficiency

gridModel_inputs = [N_monte_carlo_trials, day_hours, transformer_power_kVA, power_factor, avg_eff]

array([1.  , 0.99, 0.98, 0.97, 0.94, 0.95, 0.91, 0.88, 0.89, 0.86, 0.86,
       0.85, 0.83, 0.83, 0.84, 0.85, 0.84, 0.86, 0.83, 0.85])

In [ ]:
# Execution of probabilistic grid outage simulation from the Power_Grid() class and OutageModel_Monte_Carlo_Simulation() function

model1 = Power_Grid()
i = model1.OutageModel_Monte_Carlo_Simulation(N_monte_carlo_trials, day_hours, transformer_power_kVA, power_factor, avg_eff)
i

In [24]:
# Input parameters for solar pv power output estimation. Parameters include location latitude and longitude, age of installation and other solar panel parameters

values = list(map(float, input(f"Latitude of location \nLongitude of Location \nAge of installation \npanel Isc \nIsc temp coefficient \nPstc \nPstc temp : ").split()))

#5.7603 -0.2199 7 9.12 0.00058 270 0.0041

Latitude of location 
Longitude of Location 
Age of installation 
panel Isc 
Isc temp coefficient 
Pstc 
Pstc temp :  5.7603 -0.2199 7 9.12 0.00058 270 0.0041


In [ ]:
# Execution of solar pv power output estimation from the Solar_Model() class and the model2.Photocurrent_Power_Approximation() function

model2 = Solar_Model()
j = model2.Photocurrent_Power_Approximation(gridModel_inputs, n_day, day_hours, values, 10, Engineering_Building_panels_azimuth, Panels_per_azimuth, ghi, temp, 0.2)
j

In [10]:
# Execution of the solar pv output current estimation from the Output_Current_Voltage_Approximation() function contained in the Solar_Model() class

k = model2.Output_Current_Voltage_Approximation(4, 60, 20, Rs, Rsh, Io, n, q, K, temp)
k

array([ 0.00000000e+00, -8.07793567e-28,  0.00000000e+00, -8.07793567e-28,
        0.00000000e+00,  0.00000000e+00,  3.11506049e-02,  3.80988307e-01,
        2.06842579e+00,  3.14770960e+00,  7.69826506e+00,  1.29147566e+01,
        1.36252214e+01,  1.22592246e+01,  5.93851552e+00,  2.74353020e+00,
        9.78105974e-01,  4.01158245e-02,  0.00000000e+00, -8.07793567e-28])

In [25]:
# Input parameters for battery storage system SOC modelling and scheduling

peak_loadkW = 40
max_charge_A = 100
max_discharge_A = 30

batt_capAh = 800
terminal_volt = 100
eff = 0.8

In [ ]:
# Execution of the BESS SOC estimation and scheduling using the Battery_Energy_Storage_Model() class and the Bess_Model_Approximation function

model3 = Battery_Energy_Storage_Model()

l = model3.Bess_Model_Approximation(gridModel_inputs, pv_powerModel_inputs, pv_ivModel_inputs, batt_capAh, terminal_volt, eff, example_load_profile, peak_loadkW, max_charge_A, max_discharge_A)

l